## 10. 🆕 Consistencia de J: reponderación con nacimientos propios

**El problema (circularidad).** $J_{t+65}$ es la población de 65+ años que la ONU proyecta para el año $t+65$. Esa población está construida con los **nacimientos de la ONU** de las cohortes que la componen (nacidas entre $t-35$ y $t$). Pero en $N_t$ ya no usamos los nacimientos de la ONU: usamos DEIS (observado, hasta 2024) y, desde 2025, los escenarios propios de la sección 9. Es decir, el numerador y el denominador del $TBP$ cuentan **historias de natalidad distintas**: $N$ dice "nacieron pocos", $J$ supone "nacieron los que dijo la ONU". Si la natalidad cae, la población 65+ del futuro también tiene que ser menor, y el denominador está sobreestimado.

**Qué hace esta sección** (fragmento autónomo; depende solo de `d`, `s`, `TAU`, `RHO`, `A_OBS`, `A_LEGAL`, `DATA`, `OUT` en memoria y de los archivos que carga él mismo):

1. reconstruye $J$ como suma por edad de la población ONU y la **repondera por cohorte** con $w(c)=N_{propio}(c)/N_{ONU}(c)$;
2. compara $J$ original vs reponderado, para cada escenario;
3. recalcula $TBP_C$ y $TBP_A$ en tres versiones: (i) serie actual, (ii) $N$ propio con $J$ ONU (inconsistente), (iii) $N$ propio con $J$ reponderado (consistente);
4. exporta `tbp_cohortes_J_reponderado.csv` y verifica con asserts.

### 10.1 Carga de datos

Se cargan dos archivos de `datos/procesados/`: la población ONU por edad simple (`PopTotal`, variante media, miles) y los escenarios de natalidad de la sección 9. Rutas candidatas: `../datos/procesados`, `datos/procesados` y `DATA`; si no están, se descargan del repositorio (URL *raw*) a la carpeta de salidas. Los nacimientos ONU y DEIS salen de `s`.

In [ ]:
import os, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt
try: display
except NameError: display = print

RAW_BASE_10 = "https://raw.githubusercontent.com/carolinamsfelipe/Tp-Indicadores/main/datos/procesados"
CANDS_10 = ["../datos/procesados", "datos/procesados", DATA]

def buscar_archivo_10(nombre):
    for c in CANDS_10:
        p = os.path.join(c, nombre)
        if os.path.exists(p):
            return p
    os.makedirs(OUT, exist_ok=True)          # fallback: descarga desde GitHub (raw) a la carpeta de salidas
    p = os.path.join(OUT, nombre)
    urllib.request.urlretrieve(f"{RAW_BASE_10}/{nombre}", p)
    return p

f_pob10 = buscar_archivo_10("un_wpp2024_argentina_poblacion_edad_simple.csv")
f_esc10 = buscar_archivo_10("natalidad_escenarios_2025_2035.csv")
print("Poblacion ONU :", os.path.abspath(f_pob10)); print("Escenarios    :", os.path.abspath(f_esc10))

pop10 = pd.read_csv(f_pob10, low_memory=False)
if "Variant" in pop10.columns:
    pop10 = pop10[pop10["Variant"] == "Medium"]
# PopTotal por edad simple (columnas 0..100; 100 = grupo abierto 100+) y anio, en PERSONAS
P = pop10.pivot_table(index="Time", columns="AgeGrpStart", values="PopTotal") * 1000
P.columns = P.columns.astype(int)

esc10 = pd.read_csv(f_esc10)
ESC10 = ["persistencia", "recuperacion_lenta", "caida_adicional"]      # escenarios propios (seccion 9)
assert set(ESC10 + ["onu_medio"]) <= set(esc10.escenario.unique()), esc10.escenario.unique()
NAC10 = esc10.pivot(index="anio", columns="escenario", values="nacimientos")[ESC10 + ["onu_medio"]]

N_onu10 = s["un_births_miles"] * 1000          # nacimientos ONU (personas), 1950-2100
N_deis10 = s["deis_nacimientos"]               # nacimientos DEIS observados, 1914-2024
print("Edades en la tabla ONU:", P.columns.min(), "a", P.columns.max(), "| anios:", P.index.min(), "a", P.index.max())
print("Escenarios cargados:", ESC10, "+ onu_medio | anios", NAC10.index.min(), "a", NAC10.index.max())
display((NAC10 / 1000).round(1).loc[[2025, 2030, 2035]].rename_axis("miles de nacimientos"))

### 10.2 Método: reponderación por cohorte

La población de $a$ años en el año calendario $y$ pertenece a la cohorte $c=y-a$. Si esa cohorte tuvo, según nuestros datos, una fracción $w(c)$ de los nacimientos que supuso la ONU, asumimos que **le quedan $w(c)$ veces los sobrevivientes** que proyecta la ONU. Entonces:

$$J^{rw}(y)=\sum_{a=65}^{100+} \text{PopTotal}_{ONU}(a,y)\cdot w(y-a),\qquad w(c)=\frac{N_{propio}(c)}{N_{ONU}(c)}$$

con $N_{propio}$ = DEIS para $c\le 2024$ y el escenario para $c\ge 2025$. Se aplica a $y=t+65$ para $t=1950,\dots,2035$.

**Por qué esta forma (didáctica).** Es una regla de tres por cohorte: si una generación fue 20% más chica de lo que suponía la ONU, la población de esa generación a los 70 años también lo es. Se corrige solo lo que la circularidad contamina (los nacimientos) y se deja intacto el resto de la proyección ONU (mortalidad, migración).

**Convenciones (documentadas):**

* $w=1$ para cohortes $c<1950$: la serie `un_births_miles` empieza en 1950, no hay nacimientos ONU contra los cuales comparar.
* $w=1$ para 1971–1974: la serie DEIS tiene un hueco y el cuaderno principal completa $N_t$ con la ONU (`fuente_N`); se usa la misma regla.
* El grupo abierto **100+** es una sola fila de la tabla ONU; se le asigna $w(y-100)$ (cohorte más joven del grupo, la más numerosa). Pesa muy poco en $J$.
* La población ONU es de **mitad de año**, así que "edad $a$ en $y$" mezcla las cohortes $y-a$ e $y-a-1$. Se usa $c=y-a$ (la misma convención del cuaderno principal, donde la cohorte $t$ cumple 65 en $t+65$) y se chequea la robustez promediando ambas.

**Supuestos y límites.** (a) Mortalidad y migración de cada cohorte proporcionales a los de la proyección ONU (se reescala el tamaño, no el comportamiento). (b) Para $c\le 2024$, $w=$ DEIS/ONU mezcla la corrección por la circularidad con la **diferencia de nivel entre registro (DEIS) y estimación ONU**, que existe también en cohortes viejas (media DEIS/ONU 1950–2024: 0,97). Por eso se muestra también una variante que ajusta solo 2023–2024.

In [ ]:
AGES10 = np.arange(65, 101)            # 65..99 edades simples + 100 (= grupo abierto 100+)

def pesos_cohorte(escenario=None, solo_2023_24=False):
    """w(c) = N_propio(c) / N_ONU(c).  w=1 para c<1950 (no hay nacimientos ONU en la serie).
    N_propio = DEIS para 1950<=c<=2024 y el escenario para 2025<=c<=2035.
    Cohortes 1971-1974: la serie DEIS tiene un hueco; el cuaderno principal las completa con la ONU (fuente_N),
    asi que alli w = 1 (misma regla que N_t de d).
    escenario=None -> w = 1 en TODAS las cohortes (control: debe reproducir J de la ONU).
    solo_2023_24=True -> variante de referencia: w=1 salvo 2023 y 2024 (DEIS) y los escenarios (>=2025)."""
    w = pd.Series(1.0, index=np.arange(1900, 2101))
    if escenario is None:
        return w
    for c in range(1950, 2025):
        if ((not solo_2023_24) or c >= 2023) and not np.isnan(N_deis10.get(c, np.nan)):
            w[c] = N_deis10[c] / N_onu10[c]
    for c in range(2025, 2036):
        w[c] = NAC10.loc[c, escenario] / N_onu10[c]
    assert w.notna().all()
    return w

def J_reponderado(w, anios, mezcla=0.0):
    """J_rw(y) = sum_{a=65}^{100} PopTotal_ONU(a, y) * w(y - a).
    Convencion para el grupo abierto 100+: se le asigna w(y - 100) (la cohorte mas joven del grupo; es
    la mas numerosa y, por tamanio del grupo, el error es de segundo orden).
    mezcla=0 -> cohorte = y - a (convencion del cuaderno principal: cohorte t llega a 65 en t+65).
    mezcla=0.5 -> promedia las cohortes y-a e y-a-1 (control de robustez: la poblacion ONU es de mitad de anio)."""
    out = {}
    for y in anios:
        c = y - AGES10
        wy = (1 - mezcla) * w.reindex(c).values + mezcla * w.reindex(c - 1).values
        out[y] = float((P.loc[y, AGES10].values * wy).sum())
    return pd.Series(out)

dd = d.set_index("cohorte")
ANIOS65 = [int(c) + 65 for c in dd.index]                  # 2015..2100
J_onu10 = pd.Series(dd.J.values, index=dd.index)         # J de la ONU, indexado por cohorte t (anio t+65)

def J_rw_coh(escenario, **kw):
    r = J_reponderado(pesos_cohorte(escenario, kw.pop("solo_2023_24", False)), ANIOS65, **kw)
    r.index = dd.index
    return r

JRW = {e: J_rw_coh(e) for e in ESC10 + ["onu_medio"]}      # onu_medio = DEIS<=2024 y ONU despues
print("Pesos w(c) = N_propio/N_ONU en cohortes recientes:")
display(pd.DataFrame({"w DEIS/ONU": pesos_cohorte("persistencia").loc[[2000, 2010, 2020, 2022, 2023, 2024]],
                      }).T.round(3))
display(pd.DataFrame({e: pesos_cohorte(e).loc[2025:2035:5] for e in ESC10}).T.round(3).rename_axis("w(c), c = 2025/2030/2035"))

### 10.3 J reponderado vs J ONU

Diferencia porcentual $J^{rw}/J^{ONU}-1$. Para $t\le 2024$ todos los escenarios dan lo mismo (el stock 65+ de $t+65$ solo incluye cohortes $\le t$). Los escenarios solo afectan a las cohortes $t\ge 2025$.

In [ ]:
# Diferencia porcentual J_rw vs J_ONU, cohortes t <= 2024 (stock 2015-2089)
def difpct(Jrw, rango):
    x = 100 * (Jrw / J_onu10 - 1).loc[rango[0]:rango[1]]
    return x

rows = []
for nombre, kw in [("Método principal (DEIS 1950-2024)", {}),
                   ("Variante: solo 2023-2024 con DEIS", {"solo_2023_24": True}),
                   ("Robustez: cohorte = promedio de y-a e y-a-1", {"mezcla": 0.5})]:
    x = difpct(J_rw_coh("persistencia", **kw), (1950, 2024))
    rows.append({"variante": nombre, "media (%)": x.mean(), "media |Δ| (%)": x.abs().mean(),
                 "máx |Δ| (%)": x.abs().max(), "cohorte del máx": int(x.abs().idxmax())})
tab_dif = pd.DataFrame(rows).set_index("variante")
print(tab_dif.round(2).to_string())
x_main = difpct(JRW["persistencia"], (1950, 2024))
print(f"t <= 2024, método principal: ΔJ media = {x_main.mean():+.2f}% | media del valor absoluto = {x_main.abs().mean():.2f}% | "
      f"máx |Δ| = {x_main.abs().max():.2f}% (cohorte {int(x_main.abs().idxmax())}, J ONU {J_onu10[int(x_main.abs().idxmax())]/1e6:.2f} M -> "
      f"{JRW['persistencia'][int(x_main.abs().idxmax())]/1e6:.2f} M)")
print("Cohortes 2020-2024 (ΔJ %):", x_main.loc[2020:2024].round(2).to_dict())
post = pd.DataFrame({e: 100 * (JRW[e] / J_onu10 - 1) for e in ESC10 + ["onu_medio"]}).loc[[2025, 2030, 2035]]
print("\nΔJ % para cohortes >= 2025 (J de la cohorte t = poblacion 65+ en t+65):")
display(post.round(2))

### 10.4 Recalcular el TBP en tres versiones

$$TBP_C=\frac{N(1-\mu)\bar A\tau}{J\rho},\qquad TBP_A=\frac{TBP_C}{E_r}$$

* **(i) serie actual:** $N$ = DEIS hasta 2024 y ONU después; $J$ ONU.
* **(ii) inconsistente:** $N$ de la sección 9; $J$ ONU sin reponderar.
* **(iii) consistente:** $N$ de la sección 9; $J$ reponderado con el mismo escenario.

Se usan $\bar A=30$ (requisito legal de referencia, **no un máximo**) y $\bar A=14{,}2$ (observado ANSES). $S_{65}$, $E_{65}$, $\tau$ y $\rho$ son los de `d`.

In [ ]:
A_LIST = [("A30", A_LEGAL), ("A14_2", A_OBS)]

def N_escenario(e):
    N = dd.N_t.copy()                                     # DEIS <= 2024 ; ONU >= 2025 (serie actual)
    N.loc[2025:2035] = NAC10.loc[2025:2035, e].values
    return N

def tbp(N, J, A, tipo):
    c = N * dd.S65 * A * TAU / (J * RHO)                 # TBP_C (años)
    return c if tipo == "C" else c / dd.E65             # TBP_A = TBP_C / E_r

# (i) serie actual: N ONU post-2024, J ONU.  (ii) N seccion 9, J ONU sin reponderar (inconsistente).  (iii) N seccion 9, J reponderado.
V = {}
for A_lab, A in A_LIST:
    for tp in "CA":
        V[("i", None, A_lab, tp)] = tbp(dd.N_t, dd.J, A, tp)
        for e in ESC10:
            V[("ii", e, A_lab, tp)]  = tbp(N_escenario(e), dd.J, A, tp)
            V[("iii", e, A_lab, tp)] = tbp(N_escenario(e), JRW[e], A, tp)

# la serie (i) reproduce d
assert np.allclose(V[("i", None, "A30", "C")], dd.TBP_C_legal) and np.allclose(V[("i", None, "A14_2", "A")], dd.TBP_A_obs)
print("OK: (i) reproduce TBP_C / TBP_A de d (Ā = 30 y 14,2)")

### 10.5 Gráficos

(a) $J$ original vs reponderado en año calendario ($t+65$, 2015–2100) y su diferencia porcentual. (b) $TBP_C$ por cohorte 2000–2035 con (i), (ii) y (iii), para cada escenario y para $\bar A=30$ y $14{,}2$. La línea vertical marca el corte 2024/2025.

In [ ]:
COL10 = {"i": "#444444", "ii": "#d55e00", "iii": "#0072b2"}
ETQ_E = {"persistencia": "Persistencia (TGF 1,04)", "recuperacion_lenta": "Recuperación lenta (a 1,32)",
         "caida_adicional": "Caída adicional (a 0,95)"}
COLE = {"persistencia": "#d55e00", "recuperacion_lenta": "#009e73", "caida_adicional": "#cc79a7"}

# ---- Figura 1: J original vs reponderado (anio calendario 2015-2100) ----
anio = np.array(ANIOS65)
fig, ax = plt.subplots(1, 2, figsize=(15, 4.8))
ax[0].plot(anio, J_onu10.values / 1e6, color="#444444", lw=2.4, ls=":", label="J ONU (original)")
for e in ESC10:
    ax[0].plot(anio, JRW[e].values / 1e6, color=COLE[e], lw=1.8, label=f"J reponderado: {ETQ_E[e]}")
ax[0].set_title("Población 65+ (millones), año calendario t+65"); ax[0].set_xlabel("año calendario (t+65)")
ax[0].set_ylabel("millones"); ax[0].grid(alpha=.3); ax[0].legend(fontsize=8)
ax[0].axvline(2024 + 65 + .5, color="gray", ls="--", lw=1)
ax[0].text(2090.1, ax[0].get_ylim()[0] + 0.5, "cohortes\n≥2025", fontsize=7.5, color="gray")
for e in ESC10 + ["onu_medio"]:
    ax[1].plot(anio, 100 * (JRW[e] / J_onu10 - 1).values, color=COLE.get(e, "#444444"), lw=1.8,
               ls=":" if e == "onu_medio" else "-", label="Solo DEIS ≤2024 (ONU después)" if e == "onu_medio" else ETQ_E[e])
ax[1].axhline(0, color="gray", lw=.8); ax[1].axvline(2024 + 65 + .5, color="gray", ls="--", lw=1)
ax[1].set_title("J reponderado vs J ONU (diferencia %)"); ax[1].set_xlabel("año calendario (t+65)")
ax[1].set_ylabel("%"); ax[1].grid(alpha=.3); ax[1].legend(fontsize=8)
fig.suptitle("Sección 10: población 65+ original (ONU) vs reponderada con nacimientos propios", fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.94]); plt.savefig(f"{OUT}/seccion_10_J_reponderado.png", dpi=130); plt.show()

# ---- Figura 2: TBP_C por cohorte 2000-2035, (i), (ii), (iii) ----
fig, ax = plt.subplots(2, 3, figsize=(19, 9), sharex=True)
for i, (A_lab, A, tit) in enumerate([("A30", A_LEGAL, "Ā = 30 (requisito legal de referencia, no un máximo)"),
                                     ("A14_2", A_OBS, "Ā = 14,2 (observado ANSES)")]):
    for j, e in enumerate(ESC10):
        a = ax[i, j]
        a.plot(V[("i", None, A_lab, "C")].loc[2000:2035], color=COL10["i"], lw=2.2, ls=":", label="(i) serie actual (N ONU, J ONU)")
        a.plot(V[("ii", e, A_lab, "C")].loc[2000:2035], color=COL10["ii"], lw=1.8, label="(ii) N propio, J ONU (inconsistente)")
        a.plot(V[("iii", e, A_lab, "C")].loc[2000:2035], color=COL10["iii"], lw=2.2, label="(iii) N propio, J reponderado (consistente)")
        a.axvline(2024.5, color="gray", ls="--", lw=1)
        a.set_title(f"{ETQ_E[e]}\nTBP_C, {tit}", fontsize=9); a.grid(alpha=.3)
        if j == 0: a.set_ylabel("TBP_C (años)")
        if i == 1: a.set_xlabel("año de nacimiento de la cohorte")
        a.legend(fontsize=7.5, loc="lower left")
fig.suptitle("TBP_C por cohorte 2000–2035: serie actual vs N propio con J sin reponderar vs J reponderado", fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.95]); plt.savefig(f"{OUT}/seccion_10_TBP_C_reponderado.png", dpi=130); plt.show()

### 10.6 Tabla: cohortes 2024, 2030 y 2035

La columna `TBP_C(A30) vs (i) %` compara con la serie actual. Debajo, el efecto aislado de reponderar $J$ ((iii) vs (ii), mismo $N$) y la sensibilidad a la convención de pesos.

In [ ]:
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
filas = []
for c in [2024, 2030, 2035]:
    filas.append(dict(cohorte=c, escenario="(i) serie actual", N=dd.N_t[c], J_M=dd.J[c] / 1e6,
                      **{f"TBP_C ({l})": V[("i", None, l, "C")][c] for l, _ in A_LIST},
                      **{f"TBP_A ({l})": V[("i", None, l, "A")][c] for l, _ in A_LIST}))
    for e in ESC10:
        for ver in ["ii", "iii"]:
            Jv = dd.J[c] if ver == "ii" else JRW[e][c]
            filas.append(dict(cohorte=c, escenario=f"({ver}) {e}", N=N_escenario(e)[c], J_M=Jv / 1e6,
                              **{f"TBP_C ({l})": V[(ver, e, l, "C")][c] for l, _ in A_LIST},
                              **{f"TBP_A ({l})": V[(ver, e, l, "A")][c] for l, _ in A_LIST}))
tabla10 = pd.DataFrame(filas)
ref = {c: V[("i", None, "A30", "C")][c] for c in [2024, 2030, 2035]}
tabla10["TBP_C(A30) vs (i) %"] = [100 * (r["TBP_C (A30)"] / ref[r["cohorte"]] - 1) for _, r in tabla10.iterrows()]
display(tabla10.set_index(["cohorte", "escenario"]).round({"N": 0, "J_M": 3}).round(3))

# efecto aislado de reponderar J: (iii) vs (ii), mismo N
ef = pd.DataFrame({e: {c: 100 * (V[("iii", e, "A30", "C")][c] / V[("ii", e, "A30", "C")][c] - 1) for c in [2024, 2030, 2035]} for e in ESC10})
print("Efecto de reponderar J (iii vs ii, mismo N), TBP_C en %:"); display(ef.round(2).rename_axis("cohorte"))

# robustez del resultado (iii) a la convención de pesos, escenario persistencia, TBP_C (Ā = 30)
rob = {}
for nombre, kw in [("principal: DEIS 1950-2024", {}), ("solo 2023-2024 con DEIS", {"solo_2023_24": True}),
                   ("cohorte = prom. de y-a e y-a-1", {"mezcla": 0.5})]:
    Jv = J_rw_coh("persistencia", **kw)
    rob[nombre] = {c: tbp(N_escenario("persistencia"), Jv, A_LEGAL, "C")[c] for c in [2024, 2030, 2035]}
rob["(ii) J ONU"] = {c: V[("ii", "persistencia", "A30", "C")][c] for c in [2024, 2030, 2035]}
rob["(i) serie actual"] = {c: V[("i", None, "A30", "C")][c] for c in [2024, 2030, 2035]}
print("TBP_C (Ā=30), persistencia: sensibilidad de (iii) a la convención de pesos"); display(pd.DataFrame(rob).T.round(3).rename_axis("versión"))

### 10.7 Controles (asserts)

1. Con $w\equiv 1$ el método reproduce exactamente $J$ ONU de `d` en las 86 cohortes.
2. $J^{rw}>0$ y finito en todos los escenarios.
3. Para $t\le 2024$ todos los escenarios dan el mismo $J^{rw}$; para $t\ge 2025$ difieren.
4. Con nacimientos propios menores a los de la ONU, $J^{rw}\le J^{ONU\text{ después de }2024}$ para $t\ge 2025$.
5. (i)=(ii) hasta 2024; (iii)=(ii)·$J^{ONU}/J^{rw}$; linealidad en $\bar A$.
6. $w=1$ para $c<1950$ y 1971–74; $w\cdot N_{ONU}=N_t$ de `d` hasta 2024; un caso puntual coincide con la suma explícita.

In [ ]:
# 1) con w = 1 el método reproduce EXACTAMENTE J de la ONU en d, para todas las cohortes
J_w1 = J_reponderado(pesos_cohorte(None), ANIOS65); J_w1.index = dd.index
assert np.allclose(J_w1.values, dd.J.values, rtol=1e-12, atol=1e-6)
assert np.allclose(J_w1.values, (pop10.assign(e=pd.to_numeric(pop10.AgeGrpStart)).query("e >= 65")
                                 .groupby("Time").PopTotal.sum().loc[ANIOS65].values) * 1000)
print("OK 1) con w ≡ 1: J_rw = J de la ONU (d.J) en las 86 cohortes (y = suma PopTotal 65+ de la tabla de edad simple)")

# 2) J_rw > 0 en todos los escenarios y cohortes
for e, Jr in JRW.items():
    assert (Jr > 0).all() and np.isfinite(Jr.values).all(), e
print("OK 2) J_rw > 0 y finito en todos los escenarios y cohortes")

# 3) para t <= 2024 los escenarios no cambian J_rw (el stock 65+ de y=t+65 solo incluye cohortes c <= t <= 2024)
for e in ESC10[1:] + ["onu_medio"]:
    assert np.allclose(JRW[e].loc[:2024].values, JRW["persistencia"].loc[:2024].values, rtol=1e-12)
assert not np.allclose(JRW["recuperacion_lenta"].loc[2025:].values, JRW["persistencia"].loc[2025:].values)
print("OK 3) t <= 2024: J_rw idéntico en todos los escenarios; t >= 2025: los escenarios sí difieren")

# 4) 'onu_medio' (w = 1 desde 2025) y 'solo DEIS<=2024' coinciden con el método principal hasta t=2024 y, para t>=2025, J_rw(onu_medio) > J_rw(persistencia)
assert (JRW["onu_medio"].loc[2025:] >= JRW["persistencia"].loc[2025:] - 1e-6).all()
print("OK 4) con nacimientos propios menores a la ONU, J_rw(persistencia) <= J_rw(ONU después de 2024) para t >= 2025")

# 5) las versiones coinciden donde corresponde: (i)=(ii) para t<=2024 ; (ii)=(iii) NO coinciden (J cambia) ; (iii) usa el mismo N que (ii)
for e in ESC10:
    assert np.allclose(V[("i", None, "A30", "C")].loc[:2024], V[("ii", e, "A30", "C")].loc[:2024])
    assert np.allclose(V[("ii", e, "A30", "C")].loc[2025:] * dd.J.loc[2025:] / JRW[e].loc[2025:], V[("iii", e, "A30", "C")].loc[2025:])
assert np.allclose(V[("iii", "persistencia", "A14_2", "C")] * 30 / 14.2, V[("iii", "persistencia", "A30", "C")])   # linealidad en Ā
print("OK 5) (i)=(ii) hasta 2024; (iii)=(ii)·J_ONU/J_rw; TBP lineal en Ā")

# 6) peso w(c) = 1 para c < 1950, y la identidad sum(PopTotal·w) con w=DEIS/ONU en un caso puntual
w_p = pesos_cohorte("persistencia")
assert (w_p.loc[:1949] == 1).all() and np.isclose(w_p[2024], N_deis10[2024] / N_onu10[2024])
y0 = 2040
manual = sum(P.loc[y0, a] * w_p[y0 - a] for a in range(65, 101))
assert np.isclose(manual, J_reponderado(w_p, [y0])[y0])
assert (w_p.loc[1971:1974] == 1).all()                                    # hueco DEIS 1971-74: N_t de d usa ONU
assert np.allclose((w_p * N_onu10.reindex(w_p.index)).loc[1950:2024].values, dd.N_t.loc[1950:2024].values)   # w·N_ONU = N_t de d
print("OK 6) w(c)=1 para c<1950 y 1971-74 (hueco DEIS); w·N_ONU = N_t de d hasta 2024; cálculo puntual de J_rw(2040) coincide con la suma explícita")

### 10.8 Salida: `tbp_cohortes_J_reponderado.csv`

Formato largo, una fila por cohorte y escenario (`persistencia`, `recuperacion_lenta`, `caida_adicional` y `onu_medio` = DEIS hasta 2024 y ONU después, la referencia). Columnas: `J_onu`, `J_reponderado`, `N_propio`, y $TBP_C$ / $TBP_A$ para $\bar A=30$ y $14{,}2$ con $J$ reponderado (versión iii) y, con sufijo `_sinrepond`, con $J$ ONU (versión ii).

In [ ]:
partes = []
for e in ESC10 + ["onu_medio"]:
    t = pd.DataFrame({"cohorte": dd.index, "escenario": e, "anio_65": dd.anio_65.values,
                      "N_propio": (N_escenario(e) if e != "onu_medio" else dd.N_t).values,
                      "J_onu": dd.J.values, "J_reponderado": JRW[e].values})
    for ver, Jv, suf in [("iii", t.J_reponderado.values, ""), ("ii", t.J_onu.values, "_sinrepond")]:
        for A_lab, A in A_LIST:
            N = t.N_propio.values
            c = N * dd.S65.values * A * TAU / (Jv * RHO)
            t[f"TBP_C_{A_lab}{suf}"] = c
            t[f"TBP_A_{A_lab}{suf}"] = c / dd.E65.values
    partes.append(t)
out10 = pd.concat(partes, ignore_index=True)
assert out10.escenario.nunique() == 4 and len(out10) == 4 * len(dd)
DEST10 = next((p for p in ["../datos/procesados", "datos/procesados"] if os.path.isdir(p)), OUT)
os.makedirs(DEST10, exist_ok=True)
f10 = os.path.join(DEST10, "tbp_cohortes_J_reponderado.csv")
out10.round({"N_propio": 0, "J_onu": 0, "J_reponderado": 0}).round(6).to_csv(f10, index=False)
print("Escrito:", os.path.abspath(f10), out10.shape); print("Escrito:", f"{OUT}/seccion_10_J_reponderado.png y seccion_10_TBP_C_reponderado.png")
out10.head(3)

### 10.9 Lectura y límites

* **Qué cambia en la conclusión.** La curva post-2024 **sigue siendo más baja que la serie actual** con persistencia y con caída adicional, pero **menos** de lo que mostraba la versión inconsistente: reponderar $J$ sube $TBP_C$ de las cohortes recientes (cohorte 2035, $\bar A=30$, persistencia: 0,463 -> 0,513; serie actual 0,603). Con recuperación lenta la cohorte 2035 (0,623) queda ligeramente **por encima** de la serie actual (0,603): la ONU y ese escenario terminan en nacimientos parecidos y el $J$ menor empuja hacia arriba.
* **Tamaño del efecto.** Para $t\ge 2025$ $J$ cae hasta -9,9% (persistencia), -5,8% (recuperación) y -12,0% (caída) en la cohorte 2035. El rebote artificial de 2025 desaparece en los dos casos de natalidad baja.
* **Para $t\le 2024$** $J$ baja 2,4% en promedio (máx 4,8%, cohorte 1999), sobre todo por la diferencia de nivel DEIS-ONU en cohortes viejas, no por 2023-24; con la variante que ajusta solo 2023-24 la diferencia es 0,0% (máx 0,9%). El nivel de $TBP$ de las cohortes ≤2024 sube ~5% con el método principal y la forma de la curva no cambia.
* **Qué NO se puede decir:** que los escenarios sean pronósticos; que la reponderación sea exacta (supone mortalidad y migración proporcionales a la ONU, trata 100+ como una cohorte y usa $c=y-a$ pese a que la población es de mitad de año); que la corrección de nivel para cohortes viejas (DEIS vs ONU) sea inequívoca.
* **Pendiente:** validar los escenarios con el grupo y reemplazar el dato provisorio de 2025 del RENAPER por el definitivo.